# Discussion on Activation Functions
- Link: https://zhuanlan.zhihu.com/p/2033220796315866017

## Sigmoid
### Definition
$$
\sigma(x) = \frac{1}{1+e^{-x}} \in (0, 1)
$$
### Derivative
$$
\sigma'(x) = \sigma(x)\bigl(1 - \sigma(x)\bigr)
$$

### Pros
1. **Probability-like output** — range $(0, 1)$ makes sigmoid a natural fit for binary classification. When paired with binary cross entropy (BCE) loss

   $$
   \mathcal{L} = -\Bigl[y\log p + (1-y)\log(1-p)\Bigr],
   $$

   the gradient simplifies to $\dfrac{\partial\mathcal{L}}{\partial z} = p - y$, where $p = \sigma(z)$ and $z = w^T x + b$.

### Cons
1. **Vanishing gradient** — the derivative saturates quickly: $\max\sigma'(x) = \sigma'(0) = 0.25$. In a deep network with $l$ sigmoid layers, gradients back-propagated to early layers are scaled by at most

   $$
   (0.25)^l \to 0,
   $$

   preventing effective weight updates.

2. **Not zero-centered** — sigmoid outputs are always positive, $a \in (0, 1)$. Consider $z = w^T a + b$ where $a$ is the previous layer's sigmoid output:

   $$
   \frac{\partial z}{\partial w_i} = a_i > 0
   \quad\Rightarrow\quad
   \frac{\partial\mathcal{L}}{\partial w_i} = a_i \frac{\partial\mathcal{L}}{\partial z}.
   $$

   All weight gradients share the sign of $\dfrac{\partial\mathcal{L}}{\partial z}$, forcing **zig-zag** optimization paths that slow convergence.

### Implementation

In [4]:
import torch.nn as nn

loss_fn = nn.BCEWithLogitsLoss()  # sigmoid + BCE, numerically stable

## Tanh
### Definition
$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} \in (-1, 1)
$$
### Derivative
$$
\tanh'(x) = 1 - \tanh^2(x)
$$
### Relationship to Sigmoid
$$
\tanh(x) = 2\sigma(2x) - 1
$$

### Pros
1. **Zero-centered** — $\tanh(0) = 0$ and outputs span $(-1, 1)$. Unlike sigmoid, gradients don't all share the same sign, avoiding zig-zag optimization.

2. **Core to LSTMs** — tanh provides the candidate cell state and hidden state with both positive and negative values, while sigmoid gates control information flow (0–1):

   $$
   \begin{aligned}
   f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)} \\
   i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)} \\
   o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)} \\
   \tilde{C}_t &= \tanh(W_C [h_{t-1}, x_t] + b_C) \quad \text{(candidate cell state)} \\
   C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell state)} \\
   h_t &= o_t \odot \tanh(C_t) \quad \text{(hidden state)}
   \end{aligned}
   $$

   where $\odot$ is element-wise multiplication.

3. **Stronger gradients than sigmoid** — $\max\tanh'(x) = 1$ at $x = 0$ (vs. $0.25$). This, combined with being zero-centered, makes tanh a solid default for hidden layers in shallow multiplayer perceptrons (MLPs) on tabular data (classification & regression).

### Cons
1. **Still saturates** — $\tanh'(x) \to 0$ as $|x| \to \infty$. Better than sigmoid ($\max = 1$ vs. $0.25$), but deep networks still suffer from vanishing gradients.

2. **Expensive** — like sigmoid, tanh relies on exponentials and is much slower than ReLU/variants, making it less suitable for very deep or large-scale networks.

### Implementation

In [ ]:
import torch.nn as nn

tanh = nn.Tanh()  # zero-centered, range (-1, 1)

## Xavier Initialization
### Motivation
For sigmoid/tanh networks, poor weight initialization causes activations and gradients to vanish or explode across layers. Xavier initialization keeps their variance stable.

Why sigmoid/tanh? Near the origin both are roughly linear with derivative $\approx 1$, so the variance analysis below holds.

### Derivation
**Forward pass** — maintain activation variance:

$$
\operatorname{Var}(a^{(l)}) \approx \operatorname{Var}(a^{(l-1)})
\quad\implies\quad
\operatorname{Var}(W^{(l)}) = \frac{1}{n_{\text{in}}}
$$

**Backward pass** — maintain gradient variance:

$$
\operatorname{Var}(\delta^{(l)}) \approx \operatorname{Var}(\delta^{(l+1)})
\quad\implies\quad
\operatorname{Var}(W^{(l)}) = \frac{1}{n_{\text{out}}}
$$

**Compromise** — harmonic mean of the two constraints:

$$
\operatorname{Var}(W^{(l)}) = \frac{2}{n_{\text{in}} + n_{\text{out}}}
$$

In practice:

$$
W \sim \mathcal{U}\!\left[-\sqrt{\dfrac{6}{n_{\text{in}}+n_{\text{out}}}},\; \sqrt{\dfrac{6}{n_{\text{in}}+n_{\text{out}}}}\right]
\quad\text{or}\quad
W \sim \mathcal{N}\!\left(0,\; \dfrac{2}{n_{\text{in}}+n_{\text{out}}}\right)
$$

In [ ]:
import torch.nn as nn

layer = nn.Linear(n_in, n_out)
nn.init.xavier_uniform_(layer.weight)  # Xavier uniform init

## ReLU
### Definition
$$
\text{ReLU}(x) = \max(0, x)
$$
### Derivative
$$
\text{ReLU}'(x) = \begin{cases}
0 & x < 0 \\
1 & x > 0
\end{cases}
$$
(At $x = 0$ the subgradient is taken as $0$ in practice.)

### Pros
1. **No vanishing gradient (positive side)** — gradient is either $0$ or $1$, never decays. This enables training of very deep networks that sigmoid/tanh cannot handle.

2. **Sparse activations** — roughly 50% of neurons output zero, acting as implicit regularization and reducing co-adaptation.

3. **Computationally cheap** — just $\max(0, x)$; no exponentials or division. Much faster than sigmoid/tanh.

### Cons
1. **Dead neurons** — if a neuron receives only negative inputs, $\text{ReLU}(x) = 0$ and $\text{ReLU}'(x) = 0$, so the neuron stops learning permanently. High learning rates or poor initialization can trigger this.

2. **Not zero-centered** — outputs are $[0, \infty)$, so gradients share the same sign (like sigmoid). Partially mitigated by sparsity.

3. **Unbounded on positive side** — can lead to exploding activations in deep networks; typically handled with batch/layer normalization.

### Implementation

In [ ]:
import torch.nn as nn

relu = nn.ReLU()  # max(0, x)

## He (Kaiming) Initialization
### Motivation
ReLU zeros out negative inputs, halving the activation variance. Xavier's $\frac{1}{n_{\text{in}}}$ scaling is therefore too small — it doesn't account for the 50\% signal loss. He initialization corrects for this.

### Derivation
For ReLU, only the positive half propagates, so:

$$
\operatorname{Var}(W) = \frac{2}{n_{\text{in}}}
$$

For Leaky ReLU with negative slope $a$, both halves contribute:

$$
\operatorname{Var}(W) = \frac{2}{(1 + a^2)\,n_{\text{in}}}
$$

(When $a = 0$, this recovers the standard ReLU formula.)

In practice:

$$
W \sim \mathcal{U}\!\left[-\sqrt{\dfrac{6}{n_{\text{in}}}},\; \sqrt{\dfrac{6}{n_{\text{in}}}}\right]
\quad\text{or}\quad
W \sim \mathcal{N}\!\left(0,\; \dfrac{2}{n_{\text{in}}}\right)
$$

In [ ]:
import torch.nn as nn

layer = nn.Linear(n_in, n_out)
nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # He init

## Standard Visual Baseline
The de facto recipe for deep ConvNet / vision architectures:

> **ReLU + Kaiming init + BatchNorm**

- **ReLU** avoids vanishing gradients on the positive side
- **Kaiming init** compensates for ReLU's variance halving (half the signal is zeroed)
- **BatchNorm** stabilizes training and tames ReLU's unbounded-positive activations

This combination made very deep networks (ResNet, VGG, etc.) trainable and remains the default starting point for most vision tasks.

## Leaky ReLU
### Definition
$$
\text{LeakyReLU}(x) = \begin{cases}
x & x > 0 \\
a x & x \leq 0
\end{cases}
$$
where $a$ is a small positive constant (typically $0.01$).

### Derivative
$$
\text{LeakyReLU}'(x) = \begin{cases}
1 & x > 0 \\
a & x \leq 0
\end{cases}
$$

### Pros
1. **No dead neurons** — gradient is $a > 0$ for negative inputs, so neurons can recover even when receiving mostly negative signals.

2. **Works well for GANs** — the non-zero negative slope helps the discriminator maintain gradient flow when the generator improves, a regime where ReLU-based discriminators often collapse.

### Cons
1. **Reduced sparsity** — fewer zero activations mean less implicit regularization compared to ReLU.

2. **Extra hyperparameter** — the slope $a$ must be chosen (or learned, as in PReLU), adding complexity.

### Implementation

In [ ]:
import torch.nn as nn

leaky_relu = nn.LeakyReLU(negative_slope=0.01)  # default a = 0.01

## ELU
### Definition
$$
\text{ELU}(x) = \begin{cases}
x & x > 0 \\
\alpha(e^x - 1) & x \leq 0
\end{cases}
$$
where $\alpha > 0$ (typically $1.0$).

### Derivative
$$
\text{ELU}'(x) = \begin{cases}
1 & x > 0 \\
\alpha e^x = \text{ELU}(x) + \alpha & x \leq 0
\end{cases}
$$

### Pros
1. **Smooth at zero** — unlike ReLU, ELU is differentiable everywhere (including $x = 0$), giving cleaner gradient flow at the boundary.

2. **Near zero-centered** — $\text{ELU}(x) \to -\alpha$ as $x \to -\infty$, pulling mean activation closer to zero and reducing the zig-zag problem.

### Cons
1. **Vanishing gradient on negative side** — $\text{ELU}'(x) = \alpha e^x \to 0$ as $x \to -\infty$, so large negative inputs still saturate (soft death).

2. **Expensive** — like sigmoid/tanh, ELU requires $\exp(x)$ on the negative side, making it slower than ReLU.

### Implementation

In [ ]:
import torch.nn as nn

elu = nn.ELU(alpha=1.0)  # default alpha = 1.0

## SELU
### Definition
$$
\text{SELU}(x) = \lambda \begin{cases}
x & x > 0 \\
\alpha e^x - \alpha & x \leq 0
\end{cases}
$$
where $\lambda \approx 1.0507$, $\alpha \approx 1.6733$ are chosen so that for $X \sim \mathcal{N}(0, 1)$:

$$
\mathbb{E}[\text{SELU}(X)] = 0,\quad \operatorname{Var}[\text{SELU}(X)] = 1.
$$

This **self-normalizing** property keeps activations at zero mean and unit variance across layers without BatchNorm.

### Pros
1. **Self-normalizing** — activations automatically converge to $\mathcal{N}(0, 1)$ during training, eliminating the need for BatchNorm.

2. **Good for deep tabular MLPs** — works well on fully-connected feedforward networks for tabular data, where BatchNorm is less effective than in ConvNets.

### Cons
1. **Strict requirements** — needs (a) standardized inputs, (b) LeCun normal init $(\operatorname{Var}(W) = 1/n_{\text{in}})$, and (c) AlphaDropout instead of regular dropout. Without these, self-normalization breaks.

2. **Not for ConvNets / RNNs** — the self-normalizing property relies on the FC layer structure and doesn't transfer well to convolutions or recurrent architectures.

### Implementation

In [ ]:
import torch.nn as nn

selu = nn.SELU()  # self-normalizing, use with LeCun init + AlphaDropout

## PReLU
### Definition
$$
\text{PReLU}(x) = \begin{cases}
x & x > 0 \\
a x & x \leq 0
\end{cases}
$$
where $a$ is **learnable** (initialized to $0.25$, one per channel). Same formula as Leaky ReLU, but the slope adapts during training.

### Pros
1. **Adaptive negative slope** — the network learns how much negative signal to keep per channel, often improving accuracy over a fixed Leaky ReLU.

2. **Negligible parameter cost** — one extra scalar per channel (e.g., 64 params for a 64-channel conv layer).

### Cons
1. **Sensitive to initialization** — too small and it behaves like ReLU (dead neurons), too large and it loses sparsity.

### Implementation

In [ ]:
import torch.nn as nn

prelu = nn.PReLU(init=0.25)  # learnable negative slope

## Swish / SiLU
### Definition
$$
\text{Swish}(x) = x\,\sigma(x) = \frac{x}{1 + e^{-x}}
$$

### Derivative
$$
\text{Swish}'(x) = \sigma(x) + x\,\sigma'(x) = \sigma(x)\bigl(1 + x(1 - \sigma(x))\bigr)
$$

### Pros
1. **Self-gated** — $x$ is gated by its own sigmoid: large positives pass through, values near zero are dampened. Smooth, non-monotonic, unbounded above, bounded below (≈ $-0.28$).

2. **Shallow negative dip** — a small non-monotonic region around $x \approx -1.3$ can help optimization in deep networks.

### Cons
1. **Expensive** — requires sigmoid evaluation, though still cheaper than GELU.

### Implementation

In [ ]:
import torch.nn as nn

swish = nn.SiLU()  # PyTorch names it SiLU (= Swish)

## GELU
### Definition
$$
\text{GELU}(x) = x\,\Phi(x) = x\cdot\mathbb{P}(X \leq x),\quad X \sim \mathcal{N}(0, 1)
$$
where $\Phi$ is the standard normal CDF. A probabilistic, smooth ReLU: instead of a hard zero below $0$, GELU weights $x$ by how likely it is to be positive under a Gaussian.

### Approximation (used in practice)
$$
\text{GELU}(x) \approx 0.5x\left(1 + \tanh\!\left[\sqrt{\dfrac{2}{\pi}}\,(x + 0.044715x^3)\right]\right)
$$

### Pros
1. **Smooth & probabilistic** — outperforms ReLU in large transformer models (BERT, GPT). The smooth transition near zero improves gradient flow.

### Cons
1. **Compute-heavy** — requires $\tanh$ (or the exact CDF $\Phi$); slower than ReLU and Swish.

### Implementation

In [ ]:
import torch.nn as nn

gelu = nn.GELU()  # used in BERT, GPT, ViT

## Mish
### Definition
$$
\text{Mish}(x) = x \tanh(\text{softplus}(x)),\quad \text{softplus}(x) = \ln(1 + e^x)
$$

### Pros
1. **Smooth and self-gated** — similar to Swish but with a sharper gradient response. Often performs well in detection and segmentation models.

### Cons
1. **Most expensive** — requires $\ln$, $\exp$, and $\tanh$; slower than GELU and Swish. Rarely worth the marginal accuracy gain in production.

### Implementation

In [ ]:
import torch.nn as nn

mish = nn.Mish()  # smooth self-gated, for detection models

## Hard Swish
### Definition
$$
\text{HardSwish}(x) = x \frac{\text{ReLU6}(x + 3)}{6}
= \begin{cases}
0 & x \leq -3 \\
x & x \geq 3 \\
\dfrac{x(x + 3)}{6} & -3 < x < 3
\end{cases}
$$

### Pros
1. **Efficient Swish approximation** — replaces sigmoid with piecewise-linear ReLU6, making it much faster on mobile/edge devices. Used in MobileNetV3.

### Cons
1. **Hard saturation** — flat for $x \leq -3$ (unlike Swish's soft tail), which can slightly hurt gradient flow early in training.

### Implementation

In [ ]:
import torch.nn as nn

hard_swish = nn.Hardswish()  # MobileNetV3, efficient on edge devices

## Pretraining and fine-tuning
- Pretraining: learn general reusable knowledge from a large, broad dataset
- Fine-tuning: adapt that pretrained model to a smaller, more specific task

## Rules of thumb
- Do not change activation inside a pretrained model.
- Activation choice and initialization are linked.
- Smooth / gated activations often improve accuracy but cost compute.
- Tiny accuracy gains may not matter in products.
- There is no universally best activation.